# FDS Pokémon Battle Prediction

This notebook produces our state reconstruction submission.
It uses the code and package versions from the repository. Turn on Internet access
and attach the competition data before running it.


In [ ]:
import os
import subprocess
import sys

REPO_NAME = "FDS-Pokemon-Challenge-2026-2027"
REPO_URL = f"https://github.com/facelessM/{REPO_NAME}.git"

# While the competition is running the repo is private, so we read a GitHub
# token from Kaggle Secrets (Add-ons > Secrets, name: GITHUB_TOKEN).
# Once the repo is public this step is skipped and a plain clone is enough.
clone_url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    clone_url = f"https://{token}@github.com/facelessM/{REPO_NAME}.git"
except Exception:
    pass

if not os.path.exists(REPO_NAME):
    # capture_output keeps the token out of the notebook output
    subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_NAME],
                   check=True, capture_output=True)

os.chdir(REPO_NAME)
sys.path.insert(0, os.getcwd())
print("Running code from commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                                  capture_output=True, text=True).stdout.strip())

In [ ]:
# Install before importing the training libraries.
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                "experiments/state_reconstruction/requirements.txt"], check=True)


## Current submission

The model combines remaining HP, recovery access, uncertain P2 KO evidence and
move/team records. CV accuracy is 0.7013 across three grouped five-fold partitions.
The next cell trains on all 10,000 battles and writes the 5,000 test predictions.


In [ ]:
import hashlib
import json
from pathlib import Path

from src.data import find_data_dir

study = Path("experiments/state_reconstruction")
expected_hash = json.loads((study / "reproduction_check.json").read_text())["submission_sha256"]
output = Path("/kaggle/working/submission_state_reconstruction_deterministic.csv")
subprocess.run([sys.executable, str(study / "run.py"), "predict",
                "--data", str(find_data_dir()), "--output", str(output)], check=True)

assert hashlib.sha256(output.read_bytes()).hexdigest() == expected_hash, "Submission differs from the evaluated file"
import pandas as pd
submission = pd.read_csv(output)
print(f"Saved {len(submission)} predictions to {output}")
submission.head()


Earlier experiments remain available through `python run.py --exp <name>`.
The full comparison and rejected mixtures are in `experiments/state_reconstruction/README.md`.
